# 08 — Offline submission inference

Attach final v2 checkpoints and the exact cache_meta.json used for preprocessing. Enable a GPU; Internet off. The notebook uses the same preprocessing and model code as training and processes test studies with bounded temporary storage.

In [ ]:
import importlib.util, subprocess, sys
# Turn on only in an Internet-enabled setup session. Offline training/submission uses attached wheels.
INSTALL_MISSING = False
WHEEL_DIR = None  # optional /kaggle/input/<your-wheel-dataset>; never a guessed attachment slug
REQUIRED = ['numpy', 'pandas', 'pydicom', 'cv2', 'sklearn', 'torch', 'torchvision', 'timm', 'tqdm']
PIP_NAMES = {'cv2':'opencv-python-headless', 'sklearn':'scikit-learn'}
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing and INSTALL_MISSING:
    command = [sys.executable, '-m', 'pip', 'install']
    if WHEEL_DIR:
        command += ['--no-index', '--find-links', WHEEL_DIR]
    subprocess.check_call(command + [PIP_NAMES.get(p,p) for p in missing])
elif missing:
    raise RuntimeError(f'Missing dependencies: {missing}. Install in setup or attach wheels; see README.')
print('Required imports available. Compressed DICOM decoder availability is checked by the pilot.')


In [ ]:
from pathlib import Path
import sys, os, json, importlib
RUNTIME_ROOT = Path(os.environ.get('RSNA_V2_RUNTIME_ROOT', '/kaggle/working')) / 'rsna_v2_runtime_afb07898ab8d1af8'
PACKAGE_DIR = RUNTIME_ROOT / 'rsna_v2'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
SOURCES = {'__init__.py': '"""RSNA Knee v2 experiment toolkit, embedded into standalone Kaggle notebooks."""\n', 'common.py': '"""Frozen input contracts and provenance. No gold labels are exported for training."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\n\nimport numpy as np\nimport pandas as pd\n\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\nSLUGS = [\'acl\', \'mcl\', \'medial_meniscus\', \'lateral_meniscus\', \'medial_oa\',\n         \'lateral_oa\', \'pf_oa\', \'effusion\', \'synovitis\', \'bakers\', \'contusion\', \'fracture\']\nPSEUDO_COLS = [\'pseudo_\' + s for s in SLUGS]\nWEIGHT_COLS = [\'weight_\' + s for s in SLUGS]\nSLOTS = [\'SAG_FS\', \'SAG_NONFS\', \'COR_FS\', \'COR_NONFS\', \'AX_FS\', \'AX_NONFS\']\nUID = \'StudyInstanceUID\'\nCOMPETITION_PATHS = [\'/kaggle/input/competitions/rsna-knee-abnormality-detection\',\n                     \'/kaggle/input/rsna-knee-abnormality-detection\']\n\n\ndef hash_file(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef fingerprint(obj):\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str,\n                                     allow_nan=False).encode()).hexdigest()\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str,\n                               allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef environment():\n    packages = {}\n    for name in [\'numpy\', \'pandas\', \'torch\', \'torchvision\', \'timm\', \'pydicom\',\n                 \'opencv-python-headless\', \'scikit-learn\', \'pylibjpeg\', \'nbformat\']:\n        try:\n            packages[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            packages[name] = None\n    return {\'python\': platform.python_version(), \'platform\': platform.platform(), \'packages\': packages}\n\n\ndef resolve_unique(filename, roots, override=None):\n    if override:\n        p = Path(override)\n        if not p.is_file():\n            raise FileNotFoundError(f\'Explicit {filename} path does not exist: {p}\')\n        return p\n    hits = sorted({p.resolve() for root in roots if Path(root).exists()\n                   for p in Path(root).rglob(filename) if p.is_file()})\n    if not hits:\n        raise FileNotFoundError(f\'Attach prior output containing {filename}; searched {roots}\')\n    if len(hits) > 1 and len({hash_file(p) for p in hits}) != 1:\n        raise ValueError(f\'Conflicting copies of {filename}; set its explicit path: {hits}\')\n    return hits[0]\n\n\ndef resolve_comp_dir(explicit=None):\n    choices = [str(explicit)] if explicit else COMPETITION_PATHS\n    for p in choices:\n        if Path(p).is_dir() and (Path(p) / \'train.csv\').is_file():\n            return Path(p)\n    raise FileNotFoundError(f\'Add competition data; expected one of {choices}\')\n\n\ndef require_columns(frame, columns, context):\n    missing = set(columns) - set(frame.columns)\n    if missing:\n        raise ValueError(f\'{context}: missing columns {sorted(missing)}\')\n\n\ndef build_supervision(config):\n    """Freeze group roles, reuse v1.2 folds, and save pseudo-only train_manifest.csv."""\n    roots = config.get(\'input_roots\', [\'/kaggle/input\'])\n    paths = {key: resolve_unique(name, roots, config.get(key)) for key, name in {\n        \'slot_plan_path\': \'rsna_phase4b2a_six_slot_plan_v1.csv\',\n        \'pseudo_path\': \'rsna_pseudo_labels_v1_1_wide.csv\',\n        \'pseudo_meta_path\': \'PSEUDO_LABELS_v1_1.json\',\n    }.items()}\n    meta = json.loads(paths[\'pseudo_meta_path\'].read_text(encoding=\'utf-8-sig\'))\n    if meta.get(\'freeze_name\') != \'PSEUDO_LABELS_v1_1\':\n        raise ValueError(\'Expected unchanged PSEUDO_LABELS_v1_1 freeze\')\n    roles_raw = pd.read_csv(paths[\'slot_plan_path\'], dtype={UID: str, \'cv_group_id\': str})\n    role_cols = [UID, \'cv_group_id\', \'training_role\']\n    require_columns(roles_raw, role_cols, \'slot plan\')\n    roles = roles_raw[role_cols].drop_duplicates()\n    if roles.isna().any().any() or roles[UID].duplicated().any():\n        raise ValueError(\'Missing or inconsistent study role/group across series\')\n    roles = roles[roles.training_role.isin([\'PSEUDO_TRAIN\', \'GOLD_EVAL_LOCKED\',\n                                          \'GOLD_GROUP_LOCKED_NON_GOLD\'])].copy()\n    expected = config.get(\'expected_counts\', {\'PSEUDO_TRAIN\': 4341, \'GOLD_EVAL_LOCKED\': 58,\n                                             \'GOLD_GROUP_LOCKED_NON_GOLD\': 6})\n    counts = roles.training_role.value_counts().to_dict()\n    if expected is not None and counts != expected:\n        raise ValueError(f\'Frozen cohort mismatch: observed {counts}; expected {expected}\')\n    locked_groups = set(roles.loc[roles.training_role != \'PSEUDO_TRAIN\', \'cv_group_id\'])\n    train = roles[roles.training_role == \'PSEUDO_TRAIN\'].copy()\n    if locked_groups.intersection(train.cv_group_id):\n        raise ValueError(\'Gold/group-excluded group found in pseudo training\')\n    pseudo = pd.read_csv(paths[\'pseudo_path\'], dtype={UID: str})\n    require_columns(pseudo, [UID] + PSEUDO_COLS + WEIGHT_COLS, \'pseudo labels\')\n    if pseudo[UID].duplicated().any():\n        raise ValueError(\'Duplicate study in pseudo table\')\n    train = train.merge(pseudo[[UID] + PSEUDO_COLS + WEIGHT_COLS], on=UID,\n                        how=\'left\', validate=\'one_to_one\')\n    values = train[PSEUDO_COLS + WEIGHT_COLS].to_numpy(dtype=float)\n    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():\n        raise ValueError(\'Missing, nonfinite, or out-of-range pseudo targets/weights\')\n    if (train[WEIGHT_COLS].sum(axis=0) <= 0).any():\n        raise ValueError(\'A target has no active training supervision\')\n    folds_file = config.get(\'folds_path\')\n    if not config.get(\'regenerate_folds\', False):\n        paths[\'folds_path\'] = resolve_unique(\'rsna_phase4b1b_pseudo_fold_assignments_v1_2.csv\',\n                                            roots, folds_file)\n        folds = pd.read_csv(paths[\'folds_path\'], dtype={UID: str, \'cv_group_id\': str})\n        require_columns(folds, [UID, \'cv_group_id\', \'fold\'], \'frozen folds\')\n        if folds[UID].duplicated().any() or set(folds[UID]) != set(train[UID]):\n            raise ValueError(\'Frozen fold studies do not match the pseudo cohort exactly\')\n        train = train.merge(folds, on=[UID, \'cv_group_id\'], how=\'left\', validate=\'one_to_one\')\n    else:\n        from sklearn.model_selection import GroupKFold\n        train = train.sort_values(UID).reset_index(drop=True)\n        train[\'fold\'] = -1\n        for fold, (_, val) in enumerate(GroupKFold(n_splits=config.get(\'n_folds\', 5)).split(\n                train, groups=train.cv_group_id)):\n            train.loc[val, \'fold\'] = fold\n    if train.fold.isna().any() or (train.groupby(\'cv_group_id\').fold.nunique() != 1).any():\n        raise ValueError(\'Fold missing, group mismatch, or group split across folds\')\n    if set(train.fold) != set(range(config.get(\'n_folds\', 5))):\n        raise ValueError(\'Unexpected fold IDs\')\n    train[\'fold\'] = train.fold.astype(int)\n    for fold in sorted(train.fold.unique()):\n        for part in [train[train.fold == fold], train[train.fold != fold]]:\n            if (part[WEIGHT_COLS].sum(axis=0) <= 0).any():\n                raise ValueError(f\'Fold {fold} has a zero-weight target in train or validation\')\n    out = Path(config[\'out_dir\'])\n    input_hashes = {k: hash_file(v) for k, v in paths.items()}\n    identity = fingerprint({\'inputs\': input_hashes, \'regenerate_folds\': config.get(\'regenerate_folds\', False),\n                            \'n_folds\': config.get(\'n_folds\', 5), \'targets\': TARGETS})\n    if (out / \'supervision_meta.json\').exists():\n        previous = json.loads((out / \'supervision_meta.json\').read_text())\n        if previous[\'supervision_id\'] != identity:\n            raise ValueError(\'Output contains different supervision; choose a new out_dir\')\n    out.mkdir(parents=True, exist_ok=True)\n    train.sort_values(UID).to_csv(out / \'train_manifest.csv\', index=False)\n    roles.sort_values(UID).to_csv(out / \'study_roles.csv\', index=False)\n    write_json(out / \'supervision_meta.json\', {\n        \'supervision_id\': identity, \'freeze\': \'PSEUDO_LABELS_v1_1\', \'targets\': TARGETS,\n        \'counts\': counts, \'group_leakage_check\': \'PASS\', \'inputs\': {k: str(v) for k,v in paths.items()},\n        \'input_sha256\': input_hashes, \'train_manifest_sha256\': hash_file(out / \'train_manifest.csv\'),\n        \'study_roles_sha256\': hash_file(out / \'study_roles.csv\'),\n        \'folds_regenerated\': config.get(\'regenerate_folds\', False), \'environment\': environment(),\n        \'gold_labels_exported\': False})\n    print(f\'Supervision PASS: {counts}; {len(set(train.cv_group_id))} pseudo groups. Saved {out}\')\n    return out\n', 'preprocessing.py': '"""Deterministic, label-free MRI cache construction (not a B2B1 reproduction).\n\nOnly selected slices are retained in memory. Every source instance is decoded to\naudit failures, including instances outside the sampling band. Pixel data and\nfree-text DICOM fields are never logged. Synthetic tests do not qualify real data.\n"""\nfrom pathlib import Path\nimport csv\nimport json\nimport logging\nimport math\nimport re\nimport shutil\n\nimport numpy as np\nimport pydicom\nfrom PIL import Image\n\nfrom .common import SLOTS, fingerprint, hash_file, resolve_comp_dir, resolve_unique\n\nLOG = logging.getLogger(__name__)\nVERSION = "rsna-v2-preprocess-1"\nDEFAULTS = dict(n_slices=24, size=224, crop_mm=130.0,\n                normalization="percentile", slice_band=[.05, .95],\n                laterality="off", force_dicom=False, reject_multiframe=True,\n                orientation_tolerance_deg=2.0, slot_strategy="legacy", max_decode_failure_fraction=0.1)\nINDEX_FIELDS = ["UID", "SeriesUID", "slot", "path", "cv_group_id",\n                "training_role", "n_valid", "preprocessing_id"]\n\n\ndef _config(config):\n    config = dict(config)\n    for alias, key in (("image_size", "size"), ("norm", "normalization")):\n        if alias in config:\n            if key in config and config[key] != config[alias]:\n                raise ValueError(f"Conflicting {alias}/{key}")\n            config[key] = config[alias]\n    cfg = {**DEFAULTS, **{k: v for k, v in config.items() if k in DEFAULTS},\n           **config.get("cache", {})}\n    for key in ("n_slices", "size"):\n        if isinstance(cfg[key], bool) or int(cfg[key]) != cfg[key] or cfg[key] < 1:\n            raise ValueError(f"{key} must be a positive integer")\n        cfg[key] = int(cfg[key])\n    if cfg["normalization"] not in {"percentile", "clahe", "zscore"}:\n        raise ValueError("Unknown normalization")\n    if cfg["laterality"] not in {"off", "geometry-canonical"}:\n        raise ValueError("Unknown laterality policy")\n    if cfg["slot_strategy"] not in {"legacy", "semantic"}:\n        raise ValueError("Unknown slot strategy")\n    lo, hi = cfg["slice_band"]\n    if not 0 <= lo < hi <= 1:\n        raise ValueError("slice_band must satisfy 0 <= low < high <= 1")\n    if cfg["crop_mm"] is not None and not (math.isfinite(cfg["crop_mm"]) and cfg["crop_mm"] > 0):\n        raise ValueError("crop_mm must be positive or None")\n    if not 0 < cfg["orientation_tolerance_deg"] <= 10:\n        raise ValueError("orientation_tolerance_deg must be in (0, 10]")\n    if not 0 <= cfg[\'max_decode_failure_fraction\'] <= 1:\n        raise ValueError(\'max_decode_failure_fraction must be in [0,1]\')\n    for key in ("force_dicom", "reject_multiframe"):\n        if not isinstance(cfg[key], bool):\n            raise ValueError(f"{key} must be boolean")\n    if not cfg["reject_multiframe"]:\n        raise ValueError("Multiframe processing is not implemented; explicit rejection is required")\n    cfg["image_size"] = cfg["size"]\n    cfg["norm"] = cfg["normalization"]\n    return cfg\n\n\ndef _vector(ds, key, length):\n    try:\n        arr = np.asarray(getattr(ds, key), dtype=np.float64)\n        return arr if arr.shape == (length,) and np.isfinite(arr).all() else None\n    except (AttributeError, TypeError, ValueError):\n        return None\n\n\ndef _axes(ds):\n    v = _vector(ds, "ImageOrientationPatient", 6)\n    if v is None:\n        return None\n    x, y = v[:3], v[3:]\n    if abs(np.linalg.norm(x) - 1) > .01 or abs(np.linalg.norm(y) - 1) > .01 or abs(x @ y) > .01:\n        return None\n    x, y = x / np.linalg.norm(x), y / np.linalg.norm(y)\n    z = np.cross(x, y)\n    return np.stack([x, y, z / np.linalg.norm(z)])\n\n\ndef _atomic_json(path, obj):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef _csv(path, rows, fields):\n    temporary = path.with_suffix(".csv.tmp")\n    with temporary.open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(path)\n\n\ndef _normalize(arr, ds, cfg):\n    x = np.asarray(arr, dtype=np.float32)\n    slope, intercept = float(getattr(ds, "RescaleSlope", 1)), float(getattr(ds, "RescaleIntercept", 0))\n    if not math.isfinite(slope) or not math.isfinite(intercept) or slope == 0:\n        raise ValueError("invalid_rescale")\n    x = x * slope + intercept\n    if not np.isfinite(x).all():\n        raise ValueError("nonfinite_pixels")\n    photo = str(getattr(ds, "PhotometricInterpretation", ""))\n    if photo not in {"MONOCHROME1", "MONOCHROME2"}:\n        raise ValueError("unsupported_photometric")\n    if photo == "MONOCHROME1":\n        x = x.min() + x.max() - x\n    if cfg["normalization"] == "zscore":\n        std = float(x.std())\n        x = np.zeros_like(x) if std < 1e-6 else np.clip(((x - x.mean()) / std + 3) / 6, 0, 1)\n    else:\n        lo, hi = np.percentile(x, [1, 99])\n        if hi - lo < 1e-6:\n            lo, hi = float(x.min()), float(x.max())\n        x = np.zeros_like(x) if hi - lo < 1e-6 else np.clip((x - lo) / (hi - lo), 0, 1)\n    x = np.rint(x * 255).astype(np.uint8)\n    if cfg["normalization"] == "clahe":\n        import cv2\n        x = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(x)\n    return x\n\n\ndef _crop_resize(x, ds, cfg):\n    audit = "off"\n    if cfg["crop_mm"] is not None:\n        spacing = _vector(ds, "PixelSpacing", 2)\n        if spacing is None or (spacing <= 0).any():\n            raise ValueError("invalid_pixel_spacing_for_crop")\n        wanted = np.rint(cfg["crop_mm"] / spacing)\n        h, w = [max(1, int(min(want, actual))) for want, actual in zip(wanted, x.shape)]\n        audit = "clipped_to_image" if any(wanted > np.asarray(x.shape)) else "physical_center"\n        top, left = (x.shape[0] - h) // 2, (x.shape[1] - w) // 2\n        x = x[top:top+h, left:left+w]\n    return np.asarray(Image.fromarray(x).resize((cfg["size"], cfg["size"]), Image.Resampling.BOX)), audit\n\n\ndef preprocess_one_series(series_path, config):\n    """Return uint8 [N,H,W] and metadata containing positions/valid lists.\n\n    Invalid or padded positions are zero with valid=False. Physical positions\n    are millimetres only when position_kind=\'mm\'; otherwise they are ordinal.\n    Geometry-canonical makes cardinal in-plane axes positive in DICOM LPS;\n    this is orientation standardization, not an inferred left/right knee label.\n    """\n    cfg = _config(config)\n    folder = Path(series_path)\n    files = sorted(p for p in folder.iterdir() if p.is_file()) if folder.is_dir() else []\n    records, failures = [], []\n    for index, path in enumerate(files):\n        try:\n            ds = pydicom.dcmread(path, stop_before_pixels=True, force=cfg["force_dicom"])\n            if int(getattr(ds, "NumberOfFrames", 1)) != 1:\n                raise ValueError("multiframe_rejected")\n            records.append(dict(path=path, ds=ds, axes=_axes(ds), ipp=_vector(ds, "ImagePositionPatient", 3), index=index))\n        except Exception as exc:\n            failures.append(dict(file_index=index, stage="header", reason=type(exc).__name__, code=str(exc) if str(exc) == "multiframe_rejected" else "header_read_failed"))\n    flags = []\n    axes = next((r["axes"] for r in records if r["axes"] is not None), None)\n    consistent = axes is not None and all(r["axes"] is not None and np.all(np.sum(r["axes"] * axes, axis=1) >= math.cos(math.radians(cfg["orientation_tolerance_deg"]))) for r in records)\n    spatial = bool(records) and consistent and all(r["ipp"] is not None for r in records)\n    if spatial:\n        normal = axes[2].copy()\n        if cfg["laterality"] == "geometry-canonical" and normal[np.argmax(abs(normal))] < 0:\n            normal *= -1\n        for r in records:\n            r["position"] = float(r["ipp"] @ normal)\n        order_method, position_kind = "reference_iop_projection", "mm"\n        records.sort(key=lambda r: (r["position"], r["path"].name))\n    else:\n        flags.append("missing_or_inconsistent_geometry")\n        try:\n            values = [float(r["ds"].InstanceNumber) for r in records]\n            if not np.isfinite(values).all() or len(set(values)) != len(values):\n                raise ValueError("ambiguous_instance_numbers")\n            for r, value in zip(records, values):\n                r["position"] = value\n            records.sort(key=lambda r: (r["position"], r["path"].name))\n            order_method = "instance_number_fallback"\n        except (AttributeError, ValueError, TypeError):\n            order_method = "filename_fallback"\n        position_kind = "ordinal"\n        for index, r in enumerate(records):\n            r["position"] = float(index)\n        flags.append(order_method)\n    geometry_rejected = False\n    if not consistent and any(r["axes"] is not None for r in records) and all(r["axes"] is not None for r in records):\n        flags.append("inconsistent_orientation_rejected")\n        geometry_rejected = True\n    flips = (False, False)\n    if cfg["laterality"] == "geometry-canonical":\n        cardinal = consistent and len(set(np.argmax(abs(axes), axis=1))) == 3 and np.all(np.max(abs(axes), axis=1) >= math.cos(math.radians(cfg["orientation_tolerance_deg"])))\n        if not spatial or not cardinal:\n            flags.append("unsafe_canonical_geometry_rejected")\n            geometry_rejected = True\n        else:\n            flips = (bool(axes[1, np.argmax(abs(axes[1]))] < 0), bool(axes[0, np.argmax(abs(axes[0]))] < 0))\n    if spatial and len(records) > 1:\n        gaps = np.diff([r["position"] for r in records])\n        if (gaps <= 1e-4).any():\n            flags.append("duplicate_positions_rejected")\n            geometry_rejected = True\n        if (gaps > 0).any() and gaps.max() > 2 * np.median(gaps[gaps > 0]):\n            flags.append("large_spacing_gap")\n    for key in ("StudyInstanceUID", "SeriesInstanceUID", "FrameOfReferenceUID"):\n        values = {str(getattr(r["ds"], key, "")) for r in records}\n        if len(values) > 1:\n            flags.append(f"mixed_{key}_rejected")\n            geometry_rejected = True\n    sop = [str(getattr(r["ds"], "SOPInstanceUID", "")) for r in records]\n    if len([s for s in sop if s]) != len(set(s for s in sop if s)):\n        flags.append("duplicate_sop_rejected")\n        geometry_rejected = True\n    n, size = cfg["n_slices"], cfg["size"]\n    images = np.zeros((n, size, size), np.uint8)\n    valid = np.zeros(n, bool)\n    positions = np.zeros(n, np.float32)\n    if records:\n        lo = int(np.floor(cfg["slice_band"][0] * (len(records)-1)))\n        hi = int(np.ceil(cfg["slice_band"][1] * (len(records)-1)))\n        selected = np.rint(np.linspace(lo, hi, min(n, hi-lo+1))).astype(int)\n    else:\n        selected = []\n    selected_map = {int(r): i for i, r in enumerate(selected)}\n    crop_modes, decoded = set(), 0\n    for index, r in enumerate(records):\n        try:\n            ds = pydicom.dcmread(r["path"], force=cfg["force_dicom"])\n            raw = ds.pixel_array\n            if raw.ndim != 2 or int(getattr(ds, "SamplesPerPixel", 1)) != 1:\n                raise ValueError("not_single_grayscale_frame")\n            x = _normalize(raw, ds, cfg)\n            x, crop_mode = _crop_resize(x, ds, cfg)\n            crop_modes.add(crop_mode)\n            decoded += 1\n            if index in selected_map and not geometry_rejected:\n                out_index = selected_map[index]\n                if flips[0]:\n                    x = x[::-1, :]\n                if flips[1]:\n                    x = x[:, ::-1]\n                images[out_index] = x\n                positions[out_index] = r["position"]\n                valid[out_index] = True\n        except Exception as exc:\n            safe_codes = {"invalid_rescale", "nonfinite_pixels", "unsupported_photometric", "invalid_pixel_spacing_for_crop", "not_single_grayscale_frame"}\n            failures.append(dict(file_index=r["index"], stage="decode", reason=type(exc).__name__, code=str(exc) if str(exc) in safe_codes else "pixel_decode_or_transform_failed"))\n    metadata = dict(n_files=len(files), n_decoded=decoded, n_valid=int(valid.sum()),\n                    n_selected=len(selected), positions=positions.tolist(), valid=valid.tolist(),\n                    position_kind=position_kind, order_method=order_method,\n                    geometry_flags=flags, geometry_rejected=geometry_rejected,\n                    crop_modes=sorted(crop_modes), canonical_flips=list(flips),\n                    decode_failures=failures, complete=bool(records) and len(failures)/max(len(files),1) <= cfg[\'max_decode_failure_fraction\'] and not geometry_rejected and bool(valid.any()))\n    return images, metadata\n\n\ndef _binary(value):\n    value = str(value).strip().lower()\n    if value in {"1", "1.0", "true", "yes", "fs", "fatsat"}:\n        return 1\n    if value in {"0", "0.0", "false", "no", "nonfs", "non-fs"}:\n        return 0\n    return None\n\n\ndef _semantic_slot(row):\n    """Use explicit plane/FS metadata or unambiguous description tokens only."""\n    plane_map = {"sagittal": "SAG", "sag": "SAG", "coronal": "COR", "cor": "COR", "axial": "AX", "ax": "AX"}\n    desc = str(row.get("first_SeriesDescription", row.get("SeriesDescription", ""))).lower()\n    tokens = set(re.findall(r"[a-z0-9]+", desc))\n    planes = {plane_map[t] for t in tokens if t in plane_map}\n    explicit = plane_map.get(str(row.get("Anatomical_Plane", "")).strip().lower())\n    if explicit:\n        planes.add(explicit)\n    fs = _binary(row.get("Fat_Suppression", row.get("fat_bin", "")))\n    has_nonfs = bool(re.search(r"non[ _-]?(?:fs|fat[ _-]?sat)", desc))\n    has_fs = bool(tokens & {"fs", "fatsat", "stir", "spair"}) and not has_nonfs\n    inferred = 0 if has_nonfs else (1 if has_fs else None)\n    if len(planes) != 1 or (fs is not None and inferred is not None and fs != inferred):\n        return "UNKNOWN", "ambiguous_semantics"\n    fs = fs if fs is not None else inferred\n    if fs is None:\n        return "UNKNOWN", "unknown_fat_suppression"\n    return f"{next(iter(planes))}_{\'FS\' if fs else \'NONFS\'}", "metadata_or_description"\n\n\ndef _component(value):\n    if not value or not re.fullmatch(r"[A-Za-z0-9_.-]+", value) or value in {".", ".."}:\n        raise ValueError("Unsafe or empty source identifier")\n    return value\n\n\ndef _series_path(comp, row):\n    uid, series = _component(row["StudyInstanceUID"]), _component(row["SeriesInstanceUID"])\n    split = _component(row.get("split") or "train")\n    candidates = [comp / f"{split}_series" / uid / series, comp / f"{split}_images" / uid / series]\n    supplied = row.get("series_path", "")\n    if supplied:\n        p = Path(supplied)\n        p = p if p.is_absolute() else comp / p\n        if p.resolve().is_relative_to(comp.resolve()):\n            candidates.append(p)\n    hits = sorted({p.resolve() for p in candidates if p.is_dir()})\n    if len(hits) > 1:\n        raise ValueError("Ambiguous source series directories")\n    return hits[0] if hits else candidates[0]\n\n\ndef _source_signature(path):\n    return fingerprint([(p.name, p.stat().st_size, p.stat().st_mtime_ns)\n                        for p in sorted(path.iterdir()) if p.is_file()]) if path.is_dir() else "missing"\n\n\ndef _valid_cache(path, sidecar, preprocessing_id, source_signature, cfg):\n    try:\n        meta = json.loads(sidecar.read_text(encoding="utf-8"))\n        if meta["preprocessing_id"] != preprocessing_id or meta["source_signature"] != source_signature or meta["sha256"] != hash_file(path):\n            return None\n        with np.load(path, allow_pickle=False) as z:\n            images, valid, positions = z["images"], z["valid"], z["positions"]\n            if images.shape != (cfg["n_slices"], cfg["size"], cfg["size"]) or images.dtype != np.uint8 or valid.shape != (cfg["n_slices"],) or valid.dtype != bool or positions.shape != valid.shape or positions.dtype != np.float32 or not np.isfinite(positions).all():\n                return None\n            if int(valid.sum()) != meta["series_meta"]["n_valid"]:\n                return None\n        return meta["series_meta"]\n    except (OSError, ValueError, KeyError, EOFError):\n        return None\n\n\ndef run_preprocessing(config):\n    """Build or resume a cache. Production raises after writing incomplete audits.\n\n    Completeness means all planned series succeeded; absent legacy slots are\n    reported, not invented. smoke=True always implies complete=False globally.\n    Existing NPZs require both matching provenance and SHA256 before reuse.\n    """\n    cfg = _config(config)\n    comp = Path(resolve_comp_dir(config.get("comp_dir")))\n    roots = config.get("input_roots", ["/kaggle/input"])\n    plan = Path(resolve_unique("rsna_phase4b2a_six_slot_plan_v1.csv", roots, override=config.get("slot_plan_path")))\n    with plan.open(newline="", encoding="utf-8-sig") as f:\n        reader = csv.DictReader(f)\n        required = {"StudyInstanceUID", "SeriesInstanceUID", "slot", "cv_group_id", "training_role"}\n        if not required.issubset(reader.fieldnames or []):\n            raise ValueError(f"Slot plan missing columns: {sorted(required - set(reader.fieldnames or []))}")\n        rows = list(reader)\n    if not rows:\n        raise ValueError("Empty slot plan")\n    all_studies = sorted({r["StudyInstanceUID"] for r in rows})\n    smoke_count = config.get("smoke_studies", 0)\n    if isinstance(smoke_count, bool) or int(smoke_count) != smoke_count or smoke_count < 0:\n        raise ValueError("smoke_studies must be a nonnegative integer")\n    smoke = smoke_count > 0\n    included = set(all_studies[:int(smoke_count)] if smoke else all_studies)\n    rows = sorted([r for r in rows if r["StudyInstanceUID"] in included], key=lambda r: (r["StudyInstanceUID"], r["slot"], r["SeriesInstanceUID"]))\n    seen, study_roles = set(), {}\n    for row in rows:\n        uid = row["StudyInstanceUID"]\n        key = (uid, row["slot"])\n        if key in seen:\n            raise ValueError("Duplicate study-slot rows require an explicit upstream selection")\n        seen.add(key)\n        role = (row["cv_group_id"], row["training_role"])\n        if not all(role) or (uid in study_roles and study_roles[uid] != role):\n            raise ValueError("Missing or inconsistent study grouping/role")\n        study_roles[uid] = role\n    identity = dict(version=VERSION, config=cfg, slot_plan_sha256=hash_file(plan),\n                    smoke_studies=int(smoke_count), source_root=str(comp.resolve()))\n    preprocessing_id = fingerprint(identity)\n    out = Path(config.get("out_dir", "/kaggle/working/rsna_v2_cache"))\n    out.mkdir(parents=True, exist_ok=True)\n    config_path = out / "preprocess_config.json"\n    if config_path.exists():\n        previous = json.loads(config_path.read_text(encoding="utf-8"))\n        if previous.get("preprocessing_id") != preprocessing_id:\n            raise ValueError("Existing cache configuration/provenance differs; choose a new out_dir")\n    elif any(out.iterdir()):\n        raise ValueError("Nonempty output has no preprocessing identity; choose a new out_dir")\n    per_series = cfg["n_slices"] * cfg["size"] ** 2 + cfg["n_slices"] * 5 + 16384\n    estimate = int(len(rows) * per_series * 1.05)\n    if estimate > float(config.get("max_cache_gb", 30)) * 1e9:\n        raise OSError("Estimated cache exceeds max_cache_gb; reduce size/slices or increase explicit budget")\n    existing_bytes = sum(p.stat().st_size for p in out.rglob("*.npz"))\n    required_free = max(per_series * 2, estimate - existing_bytes) + int(config.get("disk_reserve_bytes", 256 * 1024**2))\n    if shutil.disk_usage(out).free < required_free:\n        raise OSError("Insufficient free disk space for estimated cache and reserve")\n    _atomic_json(config_path, {**identity, "preprocessing_id": preprocessing_id})\n    meta = dict(preprocessing_id=preprocessing_id, config=cfg, complete=False,\n                smoke=smoke, smoke_studies=int(smoke_count), planned_series=len(rows),\n                planned_studies=len(included), total_plan_studies=len(all_studies), estimated_bytes=estimate)\n    _atomic_json(out / "cache_meta.json", meta)\n    index, geometry, decode, slots = [], [], [], []\n    failed = 0\n    cache = out / "series"\n    cache.mkdir(exist_ok=True)\n    for number, row in enumerate(rows):\n        uid, series = row["StudyInstanceUID"], row["SeriesInstanceUID"]\n        path = _series_path(comp, row)\n        slot, slot_reason = row["slot"], "legacy_plan"\n        if cfg["slot_strategy"] == "semantic":\n            semantic_row = dict(row)\n            if not semantic_row.get("first_SeriesDescription") and path.is_dir():\n                for source_file in sorted(path.iterdir()):\n                    if not source_file.is_file():\n                        continue\n                    try:\n                        header = pydicom.dcmread(source_file, stop_before_pixels=True, force=cfg["force_dicom"], specific_tags=["SeriesDescription"])\n                        semantic_row["first_SeriesDescription"] = str(getattr(header, "SeriesDescription", ""))\n                        break\n                    except Exception:\n                        continue\n            slot, slot_reason = _semantic_slot(semantic_row)\n        slot_ok = slot in SLOTS and not any(r["UID"] == uid and r["slot"] == slot for r in index)\n        slots.append(dict(UID=uid, SeriesUID=series, legacy_slot=row["slot"], slot=slot,\n                          status=slot_reason if slot_ok else "unknown_or_ambiguous_slot"))\n        if not slot_ok:\n            failed += 1\n            continue\n        name = fingerprint([uid, series, slot]) + ".npz"\n        dest, sidecar = cache / name, cache / (name + ".json")\n        signature = _source_signature(path)\n        series_meta = _valid_cache(dest, sidecar, preprocessing_id, signature, cfg) if dest.exists() and sidecar.exists() else None\n        if series_meta is None:\n            images, series_meta = preprocess_one_series(path, cfg)\n            temporary = dest.with_suffix(".npz.tmp")\n            with temporary.open("wb") as f:\n                np.savez_compressed(f, images=images, positions=np.asarray(series_meta["positions"], np.float32), valid=np.asarray(series_meta["valid"], bool))\n            temporary.replace(dest)\n            _atomic_json(sidecar, dict(preprocessing_id=preprocessing_id, source_signature=signature,\n                                      sha256=hash_file(dest), series_meta=series_meta))\n        failed += int(not series_meta["complete"])\n        index.append(dict(UID=uid, SeriesUID=series, slot=slot, path=dest.relative_to(out).as_posix(),\n                          cv_group_id=row["cv_group_id"], training_role=row["training_role"],\n                          n_valid=series_meta["n_valid"], preprocessing_id=preprocessing_id))\n        geometry.append(dict(UID=uid, SeriesUID=series, order_method=series_meta["order_method"],\n                             position_kind=series_meta["position_kind"], n_files=series_meta["n_files"],\n                             n_decoded=series_meta["n_decoded"], n_valid=series_meta["n_valid"],\n                             flags="|".join(series_meta["geometry_flags"]), crop_modes="|".join(series_meta["crop_modes"]),\n                             complete=series_meta["complete"]))\n        decode.extend(dict(UID=uid, SeriesUID=series, **failure) for failure in series_meta["decode_failures"])\n        if number % 100 == 0:\n            LOG.info("Preprocessing %d/%d series; incomplete=%d", number+1, len(rows), failed)\n    coverage = [dict(slot=slot, planned_studies=len(included),\n                     n_studies_present=len({r["UID"] for r in index if r["slot"] == slot and r["n_valid"] > 0})) for slot in SLOTS]\n    public_index = [{(\'StudyInstanceUID\' if k == \'UID\' else \'SeriesInstanceUID\' if k == \'SeriesUID\' else k): v for k,v in row.items()} for row in index]\n    public_fields = [\'StudyInstanceUID\' if k == \'UID\' else \'SeriesInstanceUID\' if k == \'SeriesUID\' else k for k in INDEX_FIELDS]\n    _csv(out / "cache_index.csv", public_index, public_fields)\n    _csv(out / "geometry_audit.csv", geometry, ["UID", "SeriesUID", "order_method", "position_kind", "n_files", "n_decoded", "n_valid", "flags", "crop_modes", "complete"])\n    _csv(out / "decode_audit.csv", decode, ["UID", "SeriesUID", "file_index", "stage", "reason", "code"])\n    _csv(out / "slot_audit.csv", slots, ["UID", "SeriesUID", "legacy_slot", "slot", "status"])\n    _csv(out / "slot_coverage.csv", coverage, ["slot", "planned_studies", "n_studies_present"])\n    meta.update(complete=not smoke and failed == 0 and len(index) == len(rows),\n                subset_complete=failed == 0 and len(index) == len(rows), failed_series=failed,\n                cached_series=len(index), cache_index_sha256=hash_file(out / "cache_index.csv"))\n    meta["index_sha256"] = meta["cache_index_sha256"]\n    _atomic_json(out / "cache_meta.json", meta)\n    if not smoke and not meta["complete"]:\n        raise RuntimeError(f"Production cache incomplete: {failed} failed series; inspect audit CSVs")\n    return out\n', 'training.py': '"""Group-safe MRI study training, resumable checkpoints, and shared inference."""\nfrom pathlib import Path\nimport json\nimport random\nimport time\nimport shutil\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import Dataset, DataLoader\nfrom .common import (UID, TARGETS, PSEUDO_COLS, WEIGHT_COLS, SLOTS,\n                     fingerprint, hash_file, write_json, environment)\n\n\ndef load_checkpoint(path):\n    # Only load your own locally prepared experiment checkpoints.\n    return torch.load(path, map_location=\'cpu\', weights_only=False)\n\n\ndef atomic_save(data,path):\n    path=Path(path); temp=path.with_suffix(\'.tmp\'); torch.save(data,temp); temp.replace(path)\n\n\ndef macro_loss(logits,y,w,mean_weight):\n    raw=F.binary_cross_entropy_with_logits(logits.float(),y.float(),reduction=\'none\')\n    return ((raw*w).mean(0)/mean_weight).mean()\n\n\nclass StudyDataset(Dataset):\n    def __init__(self,cache_dir,index,uids,config,training=False,labels=None):\n        self.root=Path(cache_dir); self.uids=list(map(str,uids)); self.config=config\n        self.training=training; self.labels=labels\n        self.rows={str(uid):df.sort_values(\'slot\') for uid,df in index.groupby(UID)}\n        if set(self.uids)-set(self.rows): raise ValueError(\'Studies missing from cache\')\n        self.k=int(config[\'train_anchors\'] if training else config[\'eval_anchors\'])\n        self.size=int(config.get(\'image_size\',224))\n\n    def __len__(self): return len(self.uids)\n\n    def __getitem__(self,i):\n        uid=self.uids[i]; images=np.zeros((6,self.k,3,self.size,self.size),np.float32)\n        mask=np.zeros((6,self.k),bool)\n        for _,r in self.rows[uid].iterrows():\n            if r.slot not in SLOTS: raise ValueError(f\'Unsupported slot {r.slot}\')\n            s=SLOTS.index(r.slot); path=(self.root/r[\'path\']).resolve()\n            if not path.is_relative_to(self.root.resolve()): raise ValueError(\'Cache path escapes root\')\n            with np.load(path,allow_pickle=False) as f:\n                x=f[\'images\']; good=np.flatnonzero(f[\'valid\'])\n                if x.dtype!=np.uint8 or x.ndim!=3 or x.shape[1:]!=(self.size,self.size):\n                    raise ValueError(\'Cached images do not match model size or uint8 format\')\n                if len(good)==0: continue\n                if self.training and len(good)>self.k:\n                    chosen=np.sort(np.random.choice(len(good),self.k,replace=False))\n                else: chosen=np.linspace(0,len(good)-1,min(self.k,len(good))).round().astype(int)\n                for j,position in enumerate(chosen):\n                    if self.config[\'input_mode\']==\'repeat\': ids=[good[position]]*3\n                    else: ids=good[np.clip([position-1,position,position+1],0,len(good)-1)]\n                    images[s,j]=x[ids]/255.; mask[s,j]=True\n        if not mask.any(): raise ValueError(f\'All MRI inputs invalid for study {uid}\')\n        tensor=torch.from_numpy(images)\n        tensor=(tensor-tensor.new_tensor([.485,.456,.406])[None,None,:,None,None])/tensor.new_tensor([.229,.224,.225])[None,None,:,None,None]\n        result={\'uid\':uid,\'images\':tensor,\'mask\':torch.from_numpy(mask)}\n        if self.labels is not None:\n            row=self.labels.loc[uid]\n            result.update(y=torch.tensor(row[PSEUDO_COLS].to_numpy(np.float32)),\n                          w=torch.tensor(row[WEIGHT_COLS].to_numpy(np.float32)))\n        return result\n\n\ndef create_encoder(config,initialize):\n    family=config[\'model_family\']\n    if family==\'resnet18\':\n        from torchvision.models import resnet18,ResNet18_Weights\n        backbone=resnet18(weights=ResNet18_Weights.DEFAULT if initialize and config.get(\'allow_download\') and not config.get(\'pretrained_path\') else None)\n        dim=512\n    elif family==\'dinov2\':\n        import timm\n        backbone=timm.create_model(\'vit_small_patch14_dinov2.lvd142m\',pretrained=bool(initialize and config.get(\'allow_download\') and not config.get(\'pretrained_path\')),img_size=config[\'image_size\'])\n        dim=backbone.num_features\n    else: raise ValueError(\'Unknown model_family\')\n    if initialize:\n        if config.get(\'pretrained_path\'):\n            path=Path(config[\'pretrained_path\'])\n            if path.suffix==\'.safetensors\':\n                from safetensors.torch import load_file\n                state=load_file(str(path))\n            else: state=torch.load(path,map_location=\'cpu\',weights_only=True)\n            backbone.load_state_dict(state,strict=True)\n        elif not config.get(\'allow_download\'): raise ValueError(\'Attach matching pretrained weights or enable allow_download explicitly\')\n    if family==\'resnet18\': backbone.fc=nn.Identity()\n    else: backbone.reset_classifier(0)\n    return backbone,dim\n\n\nclass StudyModel(nn.Module):\n    def __init__(self,config,initialize=True):\n        super().__init__(); self.config=config\n        self.encoder,dim=create_encoder(config,initialize)\n        self.frozen=bool(config[\'frozen_encoder\'])\n        if self.frozen:\n            for p in self.encoder.parameters(): p.requires_grad=False\n        if config[\'pooling\']==\'meanmax\':\n            self.head=nn.Sequential(nn.Dropout(.2),nn.Linear(dim*2,12))\n        else:\n            self.score=nn.Linear(dim,12); self.value=nn.Linear(dim,12)\n            self.slot_embedding=nn.Embedding(6,dim)\n\n    def train(self,mode=True):\n        super().train(mode)\n        if self.frozen: self.encoder.eval()\n        return self\n\n    def forward(self,images,mask):\n        b,s,k,c,h,w=images.shape; flat=images.reshape(-1,c,h,w); active=mask.reshape(-1)\n        # Encode real slices only, so padded zeros never pollute BatchNorm.\n        features=self.encoder(flat[active]); dim=features.shape[-1]\n        all_features=features.new_zeros((len(flat),dim)); all_features[active]=features\n        x=all_features.reshape(b,s,k,dim); m=mask.unsqueeze(-1)\n        if self.config[\'pooling\']==\'meanmax\':\n            mean=(x*m).sum(2)/m.sum(2).clamp_min(1)\n            maximum=x.masked_fill(~m,-torch.inf).max(2).values\n            maximum=torch.where(torch.isfinite(maximum),maximum,torch.zeros_like(maximum))\n            series=self.head(torch.cat([mean,maximum],-1)); available=mask.any(2)\n            planes=[]; present=[]\n            for a,z in [(0,2),(2,4),(4,6)]:\n                pm=available[:,a:z].unsqueeze(-1)\n                planes.append((series[:,a:z]*pm).sum(1)/pm.sum(1).clamp_min(1))\n                present.append(available[:,a:z].any(1))\n            planes=torch.stack(planes,1); present=torch.stack(present,1).unsqueeze(-1)\n            return (planes*present).sum(1)/present.sum(1).clamp_min(1)\n        x=x+self.slot_embedding(torch.arange(6,device=x.device))[None,:,None,:]\n        scores=self.score(x).reshape(b,s*k,12).masked_fill(~mask.reshape(b,s*k,1),-torch.inf)\n        weights=scores.softmax(1)\n        return (self.value(x).reshape(b,s*k,12)*weights).sum(1)\n\n\ndef loader(cache_dir,index,uids,cfg,training=False,labels=None):\n    ds=StudyDataset(cache_dir,index,uids,cfg,training,labels)\n    return DataLoader(ds,batch_size=cfg[\'batch_size\'],shuffle=training,num_workers=cfg[\'num_workers\'],\n                      pin_memory=torch.cuda.is_available(),persistent_workers=cfg[\'num_workers\']>0)\n\n\n@torch.inference_mode()\ndef predict_studies(model,cache_dir,cache_index,uids,config,device):\n    model.eval(); output=[]\n    for batch in loader(cache_dir,cache_index,uids,config):\n        probabilities=model(batch[\'images\'].to(device),batch[\'mask\'].to(device)).sigmoid().cpu().numpy()\n        output.extend({UID:uid,**dict(zip(TARGETS,map(float,p)))} for uid,p in zip(batch[\'uid\'],probabilities))\n    return pd.DataFrame(output)\n\n\ndef load_trained_model(checkpoint,device):\n    ckpt=load_checkpoint(checkpoint)\n    if ckpt.get(\'format\')!=\'rsna_v2_1\': raise ValueError(\'Expected v2 checkpoint, not legacy archive\')\n    model=StudyModel(ckpt[\'config\'],initialize=False)\n    model.load_state_dict(ckpt[\'model\'],strict=True); model.to(device).eval()\n    return model,ckpt[\'config\']\n\n\ndef run_training(config):\n    cfg={\'model_family\':\'resnet18\',\'pooling\':\'meanmax\',\'input_mode\':\'repeat\',\'train_anchors\':8,\n         \'eval_anchors\':16,\'fold\':0,\'n_folds\':5,\'mode\':\'selection\',\'epochs\':12,\'patience\':3,\n         \'batch_size\':2,\'num_workers\':2,\'learning_rate\':1e-4,\'backbone_lr\':1e-5,\n         \'frozen_encoder\':False,\'seed\':2026,\'allow_download\':False,\'require_cuda\':True,\n         \'resume_path\':None,**config}\n    if cfg[\'pooling\'] not in [\'meanmax\',\'attention\'] or cfg[\'input_mode\'] not in [\'repeat\',\'triplet\']:\n        raise ValueError(\'Unsupported pooling or input mode\')\n    if cfg[\'mode\'] not in [\'selection\',\'refit\'] or min(cfg[\'epochs\'],cfg[\'batch_size\'],cfg[\'train_anchors\'],cfg[\'eval_anchors\'])<1:\n        raise ValueError(\'Invalid training mode or budget\')\n    if cfg[\'require_cuda\'] and not torch.cuda.is_available(): raise RuntimeError(\'Enable Kaggle GPU accelerator\')\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    random.seed(cfg[\'seed\']); np.random.seed(cfg[\'seed\']); torch.manual_seed(cfg[\'seed\'])\n    cache=Path(cfg[\'cache_dir\']); sup=Path(cfg[\'supervision_dir\']); out=Path(cfg[\'out_dir\'])\n    cm=json.loads((cache/\'cache_meta.json\').read_text()); sm=json.loads((sup/\'supervision_meta.json\').read_text())\n    if not cm[\'complete\'] or cm[\'smoke\']: raise ValueError(\'Production training requires complete non-pilot cache\')\n    if hash_file(cache/\'cache_index.csv\')!=cm[\'index_sha256\'] or hash_file(sup/\'train_manifest.csv\')!=sm[\'train_manifest_sha256\']:\n        raise ValueError(\'Input manifest hash mismatch\')\n    index=pd.read_csv(cache/\'cache_index.csv\',dtype={UID:str,\'cv_group_id\':str,\'SeriesInstanceUID\':str})\n    labels=pd.read_csv(sup/\'train_manifest.csv\',dtype={UID:str,\'cv_group_id\':str}).set_index(UID)\n    if not labels.training_role.eq(\'PSEUDO_TRAIN\').all() or labels.index.duplicated().any(): raise ValueError(\'Invalid pseudo cohort\')\n    cache_roles=index[[UID,\'cv_group_id\',\'training_role\']].drop_duplicates().set_index(UID)\n    if cache_roles.index.duplicated().any() or set(labels.index)-set(cache_roles.index): raise ValueError(\'Cache cohort inconsistency\')\n    if not cache_roles.loc[labels.index,[\'cv_group_id\',\'training_role\']].equals(labels[[\'cv_group_id\',\'training_role\']]):\n        raise ValueError(\'Cache and supervision groups/roles differ\')\n    if not index.preprocessing_id.eq(cm[\'preprocessing_id\']).all(): raise ValueError(\'Mixed cache identities\')\n    if set(labels.fold)!=set(range(cfg[\'n_folds\'])) or (labels.groupby(\'cv_group_id\').fold.nunique()!=1).any():\n        raise ValueError(\'Invalid grouped folds\')\n    train_ids=labels.index[labels.fold!=cfg[\'fold\']].tolist() if cfg[\'mode\']==\'selection\' else labels.index.tolist()\n    val_ids=labels.index[labels.fold==cfg[\'fold\']].tolist() if cfg[\'mode\']==\'selection\' else []\n    if not train_ids or (cfg[\'mode\']==\'selection\' and not val_ids): raise ValueError(\'Empty train or validation split\')\n    cfg.update(image_size=cm[\'config\'][\'size\'],preprocessing_id=cm[\'preprocessing_id\'],\n               preprocessing_config=cm[\'config\'],supervision_id=sm[\'supervision_id\'])\n    init_hash=hash_file(cfg[\'pretrained_path\']) if cfg.get(\'pretrained_path\') else \'download\'\n    identity=fingerprint({\'config\':{k:v for k,v in cfg.items() if k not in [\'out_dir\',\'resume_path\',\'cache_dir\',\'supervision_dir\',\'pretrained_path\']},\n                          \'pretrained_sha256\':init_hash,\'train_manifest\':sm[\'train_manifest_sha256\']})\n    out.mkdir(parents=True,exist_ok=True)\n    meta_path=out/\'run_meta.json\'\n    if meta_path.exists() and json.loads(meta_path.read_text())[\'run_id\']!=identity: raise ValueError(\'Output experiment mismatch\')\n    resume=load_checkpoint(cfg[\'resume_path\']) if cfg[\'resume_path\'] else None\n    if resume and resume[\'run_id\']!=identity: raise ValueError(\'Resume checkpoint experiment mismatch\')\n    model=StudyModel(cfg,initialize=resume is None).to(device)\n    head=[p for n,p in model.named_parameters() if p.requires_grad and not n.startswith(\'encoder.\')]\n    encoder=[p for p in model.encoder.parameters() if p.requires_grad]\n    groups=[{\'params\':head,\'lr\':cfg[\'learning_rate\']}]\n    if encoder: groups.append({\'params\':encoder,\'lr\':cfg[\'backbone_lr\']})\n    optimizer=torch.optim.AdamW(groups,weight_decay=1e-4)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\')\n    history=[]; best=float(\'inf\'); bad=0; start=0; best_epoch=0\n    if resume:\n        model.load_state_dict(resume[\'model\']); optimizer.load_state_dict(resume[\'optimizer\']); scaler.load_state_dict(resume[\'scaler\'])\n        history=resume[\'history\']; best=resume[\'best\']; bad=resume[\'bad\']; start=resume[\'epoch\']; best_epoch=resume[\'best_epoch\']\n        random.setstate(resume[\'rng_python\']); np.random.set_state(resume[\'rng_numpy\']); torch.set_rng_state(resume[\'rng_torch\'])\n        if device.type==\'cuda\' and resume.get(\'rng_cuda\'): torch.cuda.set_rng_state_all(resume[\'rng_cuda\'])\n        if cfg[\'mode\']==\'selection\':\n            source=Path(cfg[\'resume_path\']).parent\n            for name in [\'best.pt\',\'validation_predictions.csv\']:\n                original=source/name\n                if not original.is_file(): raise ValueError(\'Attach the complete previous run output (best.pt, last.pt, predictions) for resume\')\n                if original.resolve() != (out/name).resolve(): shutil.copyfile(original,out/name)\n            if load_checkpoint(out/\'best.pt\')[\'run_id\']!=identity: raise ValueError(\'Previous best checkpoint mismatch\')\n    state=resume\n    mean=torch.tensor(labels.loc[train_ids,WEIGHT_COLS].to_numpy(float).mean(0),device=device,dtype=torch.float32)\n    if (mean<=0).any(): raise ValueError(\'Zero active training target weight\')\n    train_loader=loader(cache,index,train_ids,cfg,True,labels)\n    meta={\'run_id\':identity,\'config\':cfg,\'complete\':False,\'mode\':cfg[\'mode\'],\n          \'supervision_id\':sm[\'supervision_id\'],\'preprocessing_id\':cm[\'preprocessing_id\'],\'environment\':environment()}\n    write_json(meta_path,meta)\n    for epoch in range(start,cfg[\'epochs\']):\n        model.train(); total=0.; n=0; begin=time.time()\n        for batch in train_loader:\n            optimizer.zero_grad(set_to_none=True)\n            images=batch[\'images\'].to(device); mask=batch[\'mask\'].to(device); y=batch[\'y\'].to(device); w=batch[\'w\'].to(device)\n            with torch.autocast(device_type=device.type,enabled=device.type==\'cuda\'):\n                loss=macro_loss(model(images,mask),y,w,mean)\n            if not torch.isfinite(loss): raise RuntimeError(\'Nonfinite training loss\')\n            scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),1.)\n            scaler.step(optimizer); scaler.update(); total+=float(loss.detach())*len(y); n+=len(y)\n        value=None\n        if val_ids:\n            pred=predict_studies(model,cache,index,val_ids,cfg,device).set_index(UID).loc[val_ids]\n            from .evaluation import weighted_soft_bce\n            value=float(weighted_soft_bce(labels.loc[val_ids,PSEUDO_COLS],pred[TARGETS],labels.loc[val_ids,WEIGHT_COLS]).mean())\n        improved=value is not None and value<best\n        if improved: best=value; bad=0; best_epoch=epoch+1\n        elif value is not None: bad+=1\n        history.append({\'epoch\':epoch+1,\'train_bce\':total/n,\'validation_bce\':value,\'seconds\':time.time()-begin})\n        state={\'format\':\'rsna_v2_1\',\'run_id\':identity,\'config\':cfg,\'model\':model.state_dict(),\n               \'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\'epoch\':epoch+1,\n               \'history\':history,\'best\':best,\'best_epoch\':best_epoch,\'bad\':bad,\n               \'rng_python\':random.getstate(),\'rng_numpy\':np.random.get_state(),\'rng_torch\':torch.get_rng_state(),\n               \'rng_cuda\':torch.cuda.get_rng_state_all() if device.type==\'cuda\' else None}\n        atomic_save(state,out/\'last.pt\')\n        if improved:\n            atomic_save(state,out/\'best.pt\'); pred.to_csv(out/\'validation_predictions.csv\')\n        pd.DataFrame(history).to_csv(out/\'history.csv\',index=False)\n        print(history[-1],flush=True)\n        if val_ids and bad>=cfg[\'patience\']: break\n    if cfg[\'mode\']==\'refit\': atomic_save(state,out/\'final.pt\')\n    elif not (out/\'best.pt\').exists():\n        raise RuntimeError(\'Resume with no new improvement requires the previous best.pt and predictions attached in output directory\')\n    meta.update(complete=True,best_epoch=best_epoch,right_censored=bool(val_ids and best_epoch==cfg[\'epochs\']),\n                checkpoint_sha256=hash_file(out/(\'best.pt\' if val_ids else \'final.pt\')))\n    if val_ids: meta[\'validation_predictions_sha256\']=hash_file(out/\'validation_predictions.csv\')\n    write_json(meta_path,meta)\n    return out\n', 'evaluation.py': '"""Pseudo-validation comparison and explicitly enabled historical gold audit."""\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score, average_precision_score\nfrom .common import (TARGETS, UID, PSEUDO_COLS, WEIGHT_COLS, hash_file,\n                     fingerprint, write_json, resolve_comp_dir)\n\n\ndef probability_table(path):\n    frame = pd.read_csv(path, dtype={UID: str})\n    if UID not in frame or frame[UID].duplicated().any():\n        raise ValueError(f\'Missing or duplicate study IDs in {path}\')\n    missing = set(TARGETS) - set(frame.columns)\n    if missing:\n        raise ValueError(f\'Probability columns missing from {path}: {missing}\')\n    p = frame[TARGETS].to_numpy(float)\n    if not np.isfinite(p).all() or (p < 0).any() or (p > 1).any():\n        raise ValueError(\'Invalid probabilities\')\n    return frame.set_index(UID)\n\n\ndef weighted_soft_bce(y, p, w):\n    y, p, w = map(lambda x: np.asarray(x, dtype=float), (y, p, w))\n    if y.shape != p.shape or y.shape != w.shape or y.ndim != 2:\n        raise ValueError(\'Target/prediction/weight shapes must agree\')\n    if not all(np.isfinite(a).all() for a in [y,p,w]) or (w < 0).any():\n        raise ValueError(\'Invalid values\')\n    if (w.sum(axis=0) <= 0).any():\n        raise ValueError(\'A target has zero active supervision\')\n    p = np.clip(p, 1e-7, 1 - 1e-7)\n    raw = -(y * np.log(p) + (1 - y) * np.log1p(-p))\n    return (raw * w).sum(axis=0) / w.sum(axis=0)\n\n\ndef compare_runs(run_dirs, supervision_dir, out_dir):\n    """Require identical validation IDs for fair comparisons. No gold labels read."""\n    if not run_dirs:\n        raise ValueError(\'Set RUN_DIRS to explicit selected training-output directories\')\n    supervision_dir = Path(supervision_dir)\n    meta = json.loads((supervision_dir / \'supervision_meta.json\').read_text())\n    manifest = supervision_dir / \'train_manifest.csv\'\n    if hash_file(manifest) != meta[\'train_manifest_sha256\']:\n        raise ValueError(\'Supervision manifest changed\')\n    labels = pd.read_csv(manifest, dtype={UID: str}).set_index(UID)\n    tables, rows = {}, []\n    common_ids = None\n    for raw_dir in run_dirs:\n        directory = Path(raw_dir)\n        frame = probability_table(directory / \'validation_predictions.csv\')\n        if not set(frame.index).issubset(labels.index):\n            raise ValueError(\'Validation predictions include non-pseudo or unknown studies\')\n        ids = sorted(frame.index)\n        if common_ids is None:\n            common_ids = ids\n        if ids != common_ids:\n            raise ValueError(\'Runs have different validation cohorts. Compare matching folds only.\')\n        # Every represented group must be held out in the frozen fold assignment.\n        folds = set(labels.loc[ids, \'fold\'])\n        expected_ids = set(labels.index[labels.fold.isin(folds)])\n        if set(ids) != expected_ids:\n            raise ValueError(\'Partial fold predictions cannot be compared as complete validation\')\n        record_path = directory / \'run_meta.json\'\n        if not record_path.is_file():\n            raise FileNotFoundError(f\'Missing training provenance: {record_path}\')\n        record = json.loads(record_path.read_text())\n        if not record.get(\'complete\') or record.get(\'mode\') != \'selection\':\n            raise ValueError(\'Only completed selection runs are eligible\')\n        if record.get(\'supervision_id\') != meta[\'supervision_id\']:\n            raise ValueError(\'Run supervision differs from comparison supervision\')\n        if \'validation_predictions_sha256\' in record and hash_file(directory / \'validation_predictions.csv\') != record[\'validation_predictions_sha256\']:\n            raise ValueError(\'Validation predictions differ from recorded hash\')\n        key = str(directory.resolve())\n        p = frame.loc[ids, TARGETS]\n        y = labels.loc[ids, PSEUDO_COLS].to_numpy(float)\n        w = labels.loc[ids, WEIGHT_COLS].to_numpy(float)\n        losses = weighted_soft_bce(y, p.to_numpy(), w)\n        tables[key] = p\n        rows.append({\'run\': key, \'n_validation\': len(ids), \'folds\': \',\'.join(map(str,sorted(folds))),\n                     \'macro_soft_bce\': float(losses.mean()),\n                     **{\'bce_\' + t: float(v) for t,v in zip(TARGETS, losses)}})\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    result = pd.DataFrame(rows).sort_values(\'macro_soft_bce\')\n    result.to_csv(out / \'pseudo_validation_comparison.csv\', index=False)\n    correlations = []\n    keys = list(tables)\n    for i,a in enumerate(keys):\n        for b in keys[i+1:]:\n            for target in TARGETS:\n                correlations.append({\'run_a\': a, \'run_b\': b, \'target\': target,\n                    \'spearman\': tables[a][target].corr(tables[b][target], method=\'spearman\')})\n    pd.DataFrame(correlations).to_csv(out / \'prediction_correlations.csv\', index=False)\n    # Equal probability mean preserves probability scale for BCE; ranking is descriptive only.\n    mean_p = sum(x.to_numpy() for x in tables.values()) / len(tables)\n    ensemble = pd.DataFrame(mean_p, index=common_ids, columns=TARGETS)\n    ensemble.index.name = UID\n    ensemble.to_csv(out / \'equal_mean_validation_predictions.csv\')\n    report = {\'n_runs\': len(tables), \'validation_studies\': len(common_ids),\n              \'supervision_id\': meta[\'supervision_id\'],\n              \'equal_mean_macro_soft_bce\': float(weighted_soft_bce(y,mean_p,w).mean()),\n              \'selection_limit\': \'Measures agreement with a report teacher; does not prove clinical or gold AUC gains.\',\n              \'gold_used\': False}\n    write_json(out / \'comparison.json\', report)\n    return result\n\n\ndef gold_metrics(labels, probabilities, bootstrap=1000, seed=2026):\n    """Descriptive historical-gold audit; report undefined targets rather than inflating macro."""\n    y = np.asarray(labels, dtype=float)\n    p = np.asarray(probabilities, dtype=float)\n    if y.shape != p.shape or y.ndim != 2 or y.shape[1] != len(TARGETS):\n        raise ValueError(\'Gold shapes must be [studies,12]\')\n    if not np.isin(y, [0,1]).all() or not np.isfinite(p).all():\n        raise ValueError(\'Gold labels must be binary and predictions finite\')\n    rows = []\n    for j,t in enumerate(TARGETS):\n        valid = len(np.unique(y[:,j])) == 2\n        rows.append({\'target\':t,\'n_positive\':int(y[:,j].sum()),\'n\':len(y),\n                     \'auc\':float(roc_auc_score(y[:,j],p[:,j])) if valid else None,\n                     \'ap\':float(average_precision_score(y[:,j],p[:,j])) if valid else None})\n    macro = float(np.mean([r[\'auc\'] for r in rows])) if all(r[\'auc\'] is not None for r in rows) else None\n    rng = np.random.default_rng(seed)\n    draws=[]\n    for _ in range(bootstrap):\n        idx = rng.integers(0,len(y),len(y))\n        if all(len(np.unique(y[idx,j])) == 2 for j in range(len(TARGETS))):\n            draws.append(np.mean([roc_auc_score(y[idx,j],p[idx,j]) for j in range(len(TARGETS))]))\n    interval = np.quantile(draws,[.025,.975]).tolist() if len(draws) >= 100 else None\n    return pd.DataFrame(rows), {\'macro_auc\':macro,\'bootstrap_95_percentile\':interval,\n        \'bootstrap_valid_draws\':len(draws),\'bootstrap_requested\':bootstrap,\n        \'limitation\':\'Repeated historical gold audit; interval does not correct adaptive model selection.\'}\n\n\ndef run_gold_audit(config):\n    if config.get(\'run_gold_audit\') is not True:\n        raise ValueError(\'Gold audit disabled. Freeze the recipe and explicitly enable once ready.\')\n    from .training import load_trained_model, predict_studies\n    import torch\n    recipe_path = Path(config[\'recipe_path\'])\n    recipe = json.loads(recipe_path.read_text())\n    if recipe.get(\'frozen\') is not True or not recipe.get(\'rationale\'):\n        raise ValueError(\'Complete and freeze the recipe before reading gold labels\')\n    checkpoints = recipe.get(\'checkpoints\', [])\n    if not checkpoints:\n        raise ValueError(\'Frozen recipe needs explicit checkpoint paths and sha256\')\n    roles = pd.read_csv(Path(config[\'supervision_dir\']) / \'study_roles.csv\', dtype={UID: str})\n    uids = sorted(roles.loc[roles.training_role == \'GOLD_EVAL_LOCKED\',UID])\n    if len(uids) != config.get(\'expected_gold\',58):\n        raise ValueError(\'Gold cohort size mismatch\')\n    cache_dir = Path(config[\'cache_dir\'])\n    cache = pd.read_csv(cache_dir / \'cache_index.csv\', dtype={UID:str})\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    all_preds=[]\n    for item in checkpoints:\n        path = Path(item[\'path\'])\n        if hash_file(path) != item[\'sha256\']:\n            raise ValueError(\'Checkpoint differs from frozen recipe\')\n        model, trained_config = load_trained_model(path,device)\n        pred = predict_studies(model,cache_dir,cache,uids,trained_config,device)\n        pred = pred.set_index(UID).loc[uids,TARGETS]\n        all_preds.append(pred.to_numpy())\n    p = np.mean(all_preds,axis=0)\n    labels = pd.read_csv(resolve_comp_dir(config.get(\'comp_dir\')) / \'train.csv\',dtype={UID:str}).set_index(UID)\n    y = labels.loc[uids,TARGETS].to_numpy(float)\n    # Bootstrap by groups if gold includes repeated groups: never mislabel a study bootstrap.\n    gold_roles = roles.set_index(UID).loc[uids]\n    if gold_roles.cv_group_id.duplicated().any():\n        raise ValueError(\'Repeated gold groups require group bootstrap; resolve before this audit\')\n    metrics, report = gold_metrics(y,p,config.get(\'bootstrap\',1000))\n    out=Path(config[\'out_dir\']); out.mkdir(parents=True,exist_ok=True)\n    metrics.to_csv(out / \'gold_metrics.csv\',index=False)\n    table=pd.DataFrame(p,columns=TARGETS); table.insert(0,UID,uids)\n    table.to_csv(out / \'gold_predictions.csv\',index=False)\n    report.update(recipe_sha256=hash_file(recipe_path),n_studies=len(uids),gold_used_for_training=False)\n    write_json(out / \'gold_audit.json\',report)\n    return metrics,report\n', 'submission.py': '"""Offline, bounded study-at-a-time inference with training preprocessing parity."""\nfrom pathlib import Path\nimport json\nimport tempfile\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom .common import UID,TARGETS,SLOTS,hash_file,resolve_comp_dir,write_json\nfrom .preprocessing import preprocess_one_series\nfrom .training import load_trained_model,predict_studies\n\n\ndef select_test_slots(series):\n    required={UID,\'SeriesInstanceUID\',\'Anatomical_Plane\',\'Fat_Suppression\'}\n    if required-set(series): raise ValueError(f\'Test series missing {required-set(series)}\')\n    result=[]\n    for _,r in series.iterrows():\n        plane=str(r.Anatomical_Plane).strip().upper()\n        aliases={\'SAGITTAL\':\'SAG\',\'CORONAL\':\'COR\',\'AXIAL\':\'AX\',\'SAG\':\'SAG\',\'COR\':\'COR\',\'AX\':\'AX\'}\n        if plane not in aliases: continue\n        fs=str(r.Fat_Suppression).strip().lower()\n        if fs in [\'1\',\'1.0\',\'true\',\'yes\']: suffix=\'FS\'\n        elif fs in [\'0\',\'0.0\',\'false\',\'no\']: suffix=\'NONFS\'\n        else: continue\n        result.append({UID:str(r[UID]),\'SeriesInstanceUID\':str(r.SeriesInstanceUID),\n                       \'slot\':aliases[plane]+\'_\'+suffix,\n                       \'n_files\':int(r.get(\'n_files\',0)),\n                       \'fluid_bin\':int(str(r.get(\'Fluid_Sensitive\',0)).lower() in [\'1\',\'1.0\',\'true\',\'yes\'])})\n    # Multiple candidates use deterministic UID tie-break; expose this explicitly in audit.\n    frame=pd.DataFrame(result,columns=[UID,\'SeriesInstanceUID\',\'slot\',\'n_files\',\'fluid_bin\'])\n    return frame.sort_values([UID,\'slot\',\'n_files\',\'fluid_bin\',\'SeriesInstanceUID\'],\n                             ascending=[True,True,False,False,True]).drop_duplicates([UID,\'slot\'])\n\n\ndef validate_submission(sample,result):\n    if set(sample.columns)!={UID,*TARGETS}: raise ValueError(\'Unexpected sample_submission target schema\')\n    if result[UID].duplicated().any() or set(result[UID])!=set(sample[UID]): raise ValueError(\'Submission UID mismatch\')\n    result=result.set_index(UID).loc[sample[UID]].reset_index()[sample.columns]\n    p=result[TARGETS].to_numpy(float)\n    if not np.isfinite(p).all() or (p<0).any() or (p>1).any(): raise ValueError(\'Nonfinite/out-of-range submission\')\n    return result\n\n\ndef run_submission(config):\n    if config.get(\'run_submission\') is not True: raise ValueError(\'Submission inference is disabled\')\n    comp=resolve_comp_dir(config.get(\'comp_dir\')); out=Path(config[\'out_dir\']); out.mkdir(parents=True,exist_ok=True)\n    path=Path(config[\'cache_meta_path\'])\n    if hash_file(path)!=config[\'cache_meta_sha256\']: raise ValueError(\'Preprocessing metadata hash mismatch\')\n    cache_meta=json.loads(path.read_text()); pcfg=cache_meta[\'config\']\n    if not cache_meta[\'complete\'] or cache_meta[\'smoke\']: raise ValueError(\'Use full production preprocessing metadata\')\n    if pcfg.get(\'slot_strategy\',\'legacy\')!=\'legacy\': raise ValueError(\'Semantic slot inference is not yet validated; use legacy slots\')\n    device=torch.device(config.get(\'device\',\'cuda\'))\n    if device.type==\'cuda\' and not torch.cuda.is_available(): raise RuntimeError(\'Enable GPU\')\n    members=[]\n    for item in config[\'checkpoints\']:\n        if hash_file(item[\'path\'])!=item[\'sha256\']: raise ValueError(\'Checkpoint hash mismatch\')\n        model,cfg=load_trained_model(item[\'path\'],device)\n        if cfg[\'preprocessing_id\']!=cache_meta[\'preprocessing_id\'] or cfg[\'preprocessing_config\']!=pcfg:\n            raise ValueError(\'Checkpoint and preprocessing cache mismatch\')\n        if cfg[\'num_workers\']!=0: cfg={**cfg,\'num_workers\':0}\n        members.append((model,cfg))\n    if not members: raise ValueError(\'Specify final checkpoint paths and hashes\')\n    sample=pd.read_csv(comp/\'sample_submission.csv\',dtype={UID:str})\n    series=pd.read_csv(comp/\'test_series.csv\',dtype={UID:str,\'SeriesInstanceUID\':str})\n    roots=[p for p in [comp/\'test_series\',comp/\'test_images\'] if p.is_dir()]\n    if len(roots)!=1: raise ValueError(\'Expected exactly one test_series or test_images directory\')\n    test_root=roots[0]\n    series[\'n_files\']=[sum(p.is_file() for p in (test_root/r[UID]/r.SeriesInstanceUID).iterdir())\n                       if (test_root/r[UID]/r.SeriesInstanceUID).is_dir() else 0 for _,r in series.iterrows()]\n    selected=select_test_slots(series); predictions=[]; audit=[]\n    for uid in sample[UID]:\n        with tempfile.TemporaryDirectory(prefix=\'rsna_v2_test_\',dir=out) as temporary:\n            root=Path(temporary); rows=[]\n            for _,r in selected[selected[UID]==uid].iterrows():\n                series_path=test_root/uid/r.SeriesInstanceUID\n                if not series_path.is_dir(): raise FileNotFoundError(series_path)\n                images,meta=preprocess_one_series(series_path,pcfg)\n                audit.append({UID:uid,\'SeriesInstanceUID\':r.SeriesInstanceUID,\'slot\':r.slot,\n                              \'n_valid\':meta[\'n_valid\'],\'decode_failures\':len(meta[\'decode_failures\'])})\n                if meta[\'n_valid\']==0: continue\n                filename=r.slot+\'.npz\'\n                np.savez_compressed(root/filename,images=images,valid=np.array(meta[\'valid\'],bool),\n                                    positions=np.array(meta[\'positions\'],np.float32))\n                rows.append({UID:uid,\'SeriesInstanceUID\':r.SeriesInstanceUID,\'slot\':r.slot,\'path\':filename})\n            if not rows: raise RuntimeError(f\'No valid test MRI inputs for {uid}\')\n            frame=pd.DataFrame(rows)\n            member_predictions=[predict_studies(model,root,frame,[uid],cfg,device)[TARGETS].to_numpy()[0]\n                                for model,cfg in members]\n            predictions.append({UID:uid,**dict(zip(TARGETS,np.mean(member_predictions,axis=0)))})\n    result=validate_submission(sample,pd.DataFrame(predictions))\n    result.to_csv(out/\'submission.csv\',index=False)\n    pd.DataFrame(audit).to_csv(out/\'test_decode_audit.csv\',index=False)\n    write_json(out/\'submission_provenance.json\',{\'checkpoints\':config[\'checkpoints\'],\n        \'cache_meta_sha256\':config[\'cache_meta_sha256\'],\'n_test_studies\':len(result),\n        \'series_selection\':\'legacy plane/FS; file count, fluid sensitivity, lexical UID tie-break\',\n        \'submission_sha256\':hash_file(out/\'submission.csv\')})\n    return out/\'submission.csv\'\n'}
for filename, source in SOURCES.items():
    (PACKAGE_DIR / filename).write_text(source, encoding='utf-8')
for name in list(sys.modules):
    if name == 'rsna_v2' or name.startswith('rsna_v2.'):
        del sys.modules[name]
sys.path.insert(0, str(RUNTIME_ROOT))
importlib.invalidate_caches()
from rsna_v2.common import *
print('Embedded source version: afb07898ab8d1af8')


In [ ]:
RUN_SUBMISSION = False
CONFIG = {
    'run_submission':RUN_SUBMISSION,
    'comp_dir':'/kaggle/input/competitions/rsna-knee-abnormality-detection',
    'out_dir':'/kaggle/working',
    'checkpoints':[],  # [{'path':'/kaggle/input/.../best.pt','sha256':'...'}]
    'cache_meta_path':None,
    'cache_meta_sha256':None,
    'device':'cuda',
}
if RUN_SUBMISSION:
    from rsna_v2.submission import run_submission
    result=run_submission(CONFIG)
    print(result)
else:
    print('Set exact checkpoint paths/hashes and preprocessing metadata, then enable RUN_SUBMISSION.')
